# 🚦 Traffic Demand Prediction — ML Pipeline

**Competition Metric:** `score = max(0, 100 * r2_score(actual, predicted))`

**Strategy:**
- Feature engineering (geohash decode, cyclical time features, geo-aggregation)
- LightGBM + XGBoost + CatBoost ensemble
- 5-Fold Out-of-Fold (OOF) cross-validation
- Optimal blend weight search


## 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully ✅")

All libraries imported successfully ✅


## 2. Load Data

In [2]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

print(f"Train shape : {train.shape}")
print(f"Test  shape : {test.shape}")
print(f"Columns     : {list(train.columns)}")
print(f"\nTarget 'demand' stats:")
print(train['demand'].describe())
print(f"\nMissing values in train:")
print(train.isnull().sum())

Train shape : (77299, 11)
Test  shape : (41778, 10)
Columns     : ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']

Target 'demand' stats:
count    7.729900e+04
mean     9.394238e-02
std      1.421905e-01
min      6.245650e-07
25%      1.822723e-02
50%      4.775994e-02
75%      1.085951e-01
max      1.000000e+00
Name: demand, dtype: float64

Missing values in train:
Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64


## 3. Quick EDA

In [3]:
# Preview data
print("=== TRAIN SAMPLE ===")
display(train.head())

print("\n=== SAMPLE SUBMISSION ===")
display(sample_sub)

print("\n=== UNIQUE VALUE COUNTS ===")
for col in train.columns:
    print(f"  {col}: {train[col].nunique()} unique values")

=== TRAIN SAMPLE ===


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy



=== SAMPLE SUBMISSION ===


,Index,demand
0,0,0.090768
1,1,0.089885
2,2,0.007037
3,3,0.079087
4,4,0.054636



=== UNIQUE VALUE COUNTS ===
  Index: 77299 unique values
  geohash: 1249 unique values
  day: 2 unique values
  timestamp: 96 unique values
  demand: 76715 unique values
  RoadType: 3 unique values
  NumberofLanes: 5 unique values
  LargeVehicles: 2 unique values
  Landmarks: 2 unique values
  Temperature: 74804 unique values
  Weather: 4 unique values


## 4. Feature Engineering

### 4a. Geohash Decoder
Geohash is a spatial encoding (e.g., `'qp09'`). We decode it into actual **lat/lng coordinates** so the model understands spatial relationships.

In [4]:
def decode_geohash_approx(gh):
    """
    Geohash base32 decode → approximate lat/lng.
    No external libs required — pure Python implementation.
    """
    BASE32 = "0123456789bcdefghjkmnpqrstuvwxyz"
    char_map = {c: i for i, c in enumerate(BASE32)}
    lat_lo, lat_hi = -90.0, 90.0
    lng_lo, lng_hi = -180.0, 180.0
    is_lng = True
    try:
        for char in str(gh).lower():
            val = char_map.get(char, 0)
            for bit in range(4, -1, -1):
                bit_val = (val >> bit) & 1
                if is_lng:
                    mid = (lng_lo + lng_hi) / 2
                    if bit_val: lng_lo = mid
                    else: lng_hi = mid
                else:
                    mid = (lat_lo + lat_hi) / 2
                    if bit_val: lat_lo = mid
                    else: lat_hi = mid
                is_lng = not is_lng
    except:
        return 0.0, 0.0
    return (lat_lo + lat_hi) / 2, (lng_lo + lng_hi) / 2

# Quick test
print("Decoded 'qp09' →", decode_geohash_approx("qp09"))

Decoded 'qp09' → (-5.361328125, 90.87890625)


### 4b. Full Feature Engineering Function

Key engineered features:
| Feature | Why |
|---------|-----|
| `geo_lat`, `geo_lng` | Raw spatial coords from geohash |
| `geo_prefix` | First 4 chars = ~40km area cluster |
| `ts_hour_sin/cos` | **Cyclical** encoding — hour 23 ≈ hour 0 |
| `day_sin/cos` | Cyclical day-of-week |
| `is_weekend` | Binary flag |
| `time_bucket` | Rush hour / night / etc. |
| `hour_x_day` | Interaction: hour × day |

In [5]:
def feature_engineer(df):
    df = df.copy()

    # --- Geohash features ---
    if "geohash" in df.columns:
        coords = df["geohash"].apply(decode_geohash_approx)
        df["geo_lat"] = coords.apply(lambda x: x[0])
        df["geo_lng"] = coords.apply(lambda x: x[1])
        df["geohash_len"] = df["geohash"].apply(lambda x: len(str(x)))
        # Precision prefix clusters (first 4 chars = ~40km box)
        df["geo_prefix"] = df["geohash"].apply(lambda x: str(x)[:4])

    # --- Timestamp features ---
    if "timestamp" in df.columns:

        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

        df["ts_hour"] = df["timestamp"].dt.hour.fillna(0)
        df["ts_minute"] = df["timestamp"].dt.minute.fillna(0)

        # Cyclical encoding — captures periodicity of time
        df["ts_hour_sin"] = np.sin(2 * np.pi * df["ts_hour"] / 24)
        df["ts_hour_cos"] = np.cos(2 * np.pi * df["ts_hour"] / 24)
        df["ts_min_sin"]  = np.sin(2 * np.pi * df["ts_minute"] / 60)
        df["ts_min_cos"]  = np.cos(2 * np.pi * df["ts_minute"] / 60)

        # Time-of-day buckets
        df["time_bucket"] = pd.cut(
            df["ts_hour"],
            bins=[-1, 5, 9, 12, 14, 17, 20, 23],
            labels=["night", "morning_rush", "midday", "lunch",
                    "afternoon_rush", "evening", "late"]
        )

    # --- Day features ---
    if "day" in df.columns:
        df["day"] = df["day"].astype(float)
        df["day_sin"]    = np.sin(2 * np.pi * df["day"] / 7)
        df["day_cos"]    = np.cos(2 * np.pi * df["day"] / 7)
        df["is_weekend"] = (df["day"] % 7 >= 5).astype(int)

    # --- Interaction features ---
    if "ts_hour" in df.columns and "day" in df.columns:
        df["hour_x_day"]   = df["ts_hour"] * df["day"]
        df["hour_x_lanes"] = df["ts_hour"] * df.get("NumberofLanes", 0)

    df.drop("timestamp", axis=1, inplace=True, errors="ignore")

    return df

# Apply to both datasets
train = feature_engineer(train)
test  = feature_engineer(test)
print(f"Train shape after feature engineering: {train.shape}")
print(f"Test  shape after feature engineering: {test.shape}")

Train shape after feature engineering: (77299, 26)
Test  shape after feature engineering: (41778, 25)


### 4c. Geo-Aggregation Features (Target Statistics)

For each geohash location, we compute demand statistics **from training data only** (no leakage).
These are the most powerful features — they encode the **baseline demand level of each location**.

In [6]:
print("Building geo-aggregation features...")

# Per-geohash demand stats
if "geohash" in train.columns and "demand" in train.columns:
    geo_stats = train.groupby("geohash")["demand"].agg(
        geo_mean_demand="mean",
        geo_median_demand="median",
        geo_std_demand="std",
        geo_max_demand="max",
        geo_count="count"
    ).reset_index()

    train = train.merge(geo_stats, on="geohash", how="left")
    test  = test.merge(geo_stats, on="geohash", how="left")

    # Prefix-level stats (broader area)
    if "geo_prefix" in train.columns:
        pfx_stats = train.groupby("geo_prefix")["demand"].agg(
            pfx_mean_demand="mean",
            pfx_std_demand="std"
        ).reset_index()
        train = train.merge(pfx_stats, on="geo_prefix", how="left")
        test  = test.merge(pfx_stats, on="geo_prefix", how="left")

# Hour-level demand averages
if "ts_hour" in train.columns:
    hr_stats = train.groupby("ts_hour")["demand"].agg(
        hr_mean_demand="mean",
        hr_std_demand="std"
    ).reset_index()
    train = train.merge(hr_stats, on="ts_hour", how="left")
    test  = test.merge(hr_stats, on="ts_hour", how="left")

print(f"Train shape after aggregation features: {train.shape}")

Building geo-aggregation features...
Train shape after aggregation features: (77299, 35)


### 4d. Encode Categorical Columns

In [7]:
CAT_COLS = ["RoadType", "Weather", "time_bucket", "geo_prefix", "geohash"]
CAT_COLS = [c for c in CAT_COLS if c in train.columns]

for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([train[col].astype(str), test[col].astype(str)])
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

# Binary columns
for col in ["LargeVehicles", "Landmarks"]:
    if col in train.columns:
        mapping = {True: 1, False: 0, "True": 1, "False": 0,
                   1: 1, 0: 0, "Yes": 1, "No": 0}
        train[col] = train[col].map(mapping).fillna(0)
        test[col]  = test[col].map(mapping).fillna(0)

print("Encoding done ✅")
print(f"Categorical columns encoded: {CAT_COLS}")

Encoding done ✅
Categorical columns encoded: ['RoadType', 'Weather', 'time_bucket', 'geo_prefix', 'geohash']


## 5. Prepare Feature Matrix

In [8]:
DROP_COLS = [c for c in ["Index", "demand"] if c in train.columns]

X      = train.drop(columns=DROP_COLS)
y      = train["demand"]
X_test = test.drop(columns=[c for c in ["Index"] if c in test.columns])

# Align test columns to train
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# Fill remaining NaNs with median
X      = X.fillna(X.median(numeric_only=True))
X_test = X_test.fillna(X_test.median(numeric_only=True))

print(f"X shape      : {X.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"\nAll features:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

X shape      : (77299, 33)
X_test shape : (41778, 33)

All features:
   1. geohash
   2. day
   3. RoadType
   4. NumberofLanes
   5. LargeVehicles
   6. Landmarks
   7. Temperature
   8. Weather
   9. geo_lat
  10. geo_lng
  11. geohash_len
  12. geo_prefix
  13. ts_hour
  14. ts_minute
  15. ts_hour_sin
  16. ts_hour_cos
  17. ts_min_sin
  18. ts_min_cos
  19. time_bucket
  20. day_sin
  21. day_cos
  22. is_weekend
  23. hour_x_day
  24. hour_x_lanes
  25. geo_mean_demand
  26. geo_median_demand
  27. geo_std_demand
  28. geo_max_demand
  29. geo_count
  30. pfx_mean_demand
  31. pfx_std_demand
  32. hr_mean_demand
  33. hr_std_demand


## 6. Model Training — 5-Fold OOF CV

Each model is trained using **5-Fold Out-of-Fold** cross-validation:
- Each fold's validation set is predicted by a model that **never saw it**
- Test predictions are averaged across all 5 folds
- This gives unbiased performance estimates and reduces overfitting

### 6a. LightGBM

In [9]:
# Remove timestamp from training features
if 'timestamp' in X.columns:
    X = X.drop(columns=['timestamp'])

if 'timestamp' in X_test.columns:
    X_test = X_test.drop(columns=['timestamp'])

print(X.select_dtypes(include=['object', 'string']).columns.tolist())

[]


In [10]:
N_FOLDS = 5

kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=42
)

oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))

lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 127,
    "learning_rate": 0.03,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
    "min_child_samples": 20,
    "n_estimators": 2000,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

print("Training LightGBM...")
print("=" * 50)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):

    X_tr = X.iloc[tr_idx]
    X_val = X.iloc[val_idx]

    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]

    model = lgb.LGBMRegressor(**lgb_params)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(100, verbose=False)
        ]
    )

    val_pred = model.predict(X_val)

    oof_lgb[val_idx] = val_pred

    test_lgb += model.predict(X_test) / N_FOLDS

    r2 = r2_score(y_val, val_pred)

    print(
        f"Fold {fold}: "
        f"R² = {r2:.5f}"
    )

overall_lgb = r2_score(y, oof_lgb)

print("\nLightGBM OOF R²:", overall_lgb)
print("Competition Score:", max(0, 100 * overall_lgb))

Training LightGBM...
Fold 1: R² = 0.95783
Fold 2: R² = 0.95824
Fold 3: R² = 0.95925
Fold 4: R² = 0.95512
Fold 5: R² = 0.95909

LightGBM OOF R²: 0.9579402296129227
Competition Score: 95.79402296129227


### 6b. XGBoost

In [11]:
oof_xgb  = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "max_depth": 7,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "min_child_weight": 5,
    "early_stopping_rounds": 100,
    "random_state": 42,
    "n_jobs": -1,
    "tree_method": "hist",
    "verbosity": 0,
}

print("Training XGBoost...")
print("=" * 50)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    oof_xgb[val_idx]  = model.predict(X_val)
    test_xgb         += model.predict(X_test) / N_FOLDS

    r2    = r2_score(y_val, oof_xgb[val_idx])
    score = max(0, 100 * r2)
    print(f"  Fold {fold+1} | R²={r2:.4f} | Score={score:.2f}")

overall_xgb = max(0, 100 * r2_score(y, oof_xgb))
print(f"\n  ✅ XGBoost OOF Score: {overall_xgb:.4f}")

Training XGBoost...
  Fold 1 | R²=0.9590 | Score=95.90
  Fold 2 | R²=0.9588 | Score=95.88
  Fold 3 | R²=0.9605 | Score=96.05
  Fold 4 | R²=0.9551 | Score=95.51
  Fold 5 | R²=0.9595 | Score=95.95

  ✅ XGBoost OOF Score: 95.8619


### 6c. CatBoost

In [12]:
oof_cat  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))

print("Training CatBoost...")
print("=" * 50)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3,
        bagging_temperature=0.5,
        random_strength=0.5,
        early_stopping_rounds=100,
        eval_metric="RMSE",
        random_seed=42,
        verbose=False,
    )
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)

    oof_cat[val_idx]  = model.predict(X_val)
    test_cat         += model.predict(X_test) / N_FOLDS

    r2    = r2_score(y_val, oof_cat[val_idx])
    score = max(0, 100 * r2)
    print(f"  Fold {fold+1} | R²={r2:.4f} | Score={score:.2f}")

overall_cat = max(0, 100 * r2_score(y, oof_cat))
print(f"\n  ✅ CatBoost OOF Score: {overall_cat:.4f}")

Training CatBoost...
  Fold 1 | R²=0.9577 | Score=95.77
  Fold 2 | R²=0.9567 | Score=95.67
  Fold 3 | R²=0.9584 | Score=95.84
  Fold 4 | R²=0.9532 | Score=95.32
  Fold 5 | R²=0.9570 | Score=95.70

  ✅ CatBoost OOF Score: 95.6619


## 7. Optimal Ensemble Blending

Grid-search over all (w_LGB, w_XGB, w_CAT) combinations where weights sum to 1.
We pick the weights that maximize OOF R² — using OOF preds means **no data leakage**.

In [13]:
print("Searching for optimal blend weights...")
print("=" * 50)

best_score   = -np.inf
best_weights = (1/3, 1/3, 1/3)

for w1 in np.arange(0, 1.01, 0.05):
    for w2 in np.arange(0, 1.01 - w1, 0.05):
        w3 = round(1 - w1 - w2, 6)
        if w3 < 0:
            continue
        blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
        score = r2_score(y, blend)
        if score > best_score:
            best_score   = score
            best_weights = (w1, w2, w3)

w1, w2, w3 = best_weights
print(f"Best weights  : LGB={w1:.2f} | XGB={w2:.2f} | CAT={w3:.2f}")
print(f"Ensemble OOF  : R²={best_score:.4f}")
print(f"Competition Score Estimate: {max(0, 100*best_score):.2f}")

# Compare individual vs ensemble
print("\n--- Model Comparison ---")
print(f"  LightGBM alone : {overall_lgb:.2f}")
print(f"  XGBoost alone  : {overall_xgb:.2f}")
print(f"  CatBoost alone : {overall_cat:.2f}")
print(f"  Ensemble       : {max(0, 100*best_score):.2f}  ← best")

Searching for optimal blend weights...
Best weights  : LGB=0.25 | XGB=0.50 | CAT=0.25
Ensemble OOF  : R²=0.9591
Competition Score Estimate: 95.91

--- Model Comparison ---
  LightGBM alone : 0.96
  XGBoost alone  : 95.86
  CatBoost alone : 95.66
  Ensemble       : 95.91  ← best


## 8. Generate Submission File

In [14]:
# Final test predictions using optimal weights
final_preds = w1 * test_lgb + w2 * test_xgb + w3 * test_cat

# Clip to valid range (demand must be >= 0)
final_preds = np.clip(final_preds, 0, None)

# Build submission dataframe
index_col = test["Index"] if "Index" in test.columns else pd.RangeIndex(len(test))

submission = pd.DataFrame({
    "Index": index_col,
    "demand": final_preds
})

print(f"Submission shape : {submission.shape}  (expected 41778 x 2)")
print(f"\nFirst 5 rows:")
submission.head()

print(f"\nPrediction statistics:")
print(submission["demand"].describe())

# Save
submission.to_csv("submission_1.csv", index=False)
print("\n✅ submission_1.csv saved successfully!")
print(f"🏆 Estimated competition score: {max(0, 100*best_score):.2f}")

Submission shape : (41778, 2)  (expected 41778 x 2)

First 5 rows:

Prediction statistics:
count    41778.000000
mean         0.124881
std          0.167057
min          0.000946
25%          0.028495
50%          0.063340
75%          0.135633
max          1.132710
Name: demand, dtype: float64

✅ submission_1.csv saved successfully!
🏆 Estimated competition score: 95.91
